# Propensity Modeling & Segmentation
- Justin Wall, Data Scientist, Antler Industrial Supply

## Data Import

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_PATH = "../data/antler_industrial_accounts.csv"  # adjust to your path
df = pd.read_csv(DATA_PATH)

print(df.shape)
print("duplicate ids:", df.account_id.duplicated().sum())
print("nulls:", df.isna().sum().sum())
print("Number of Regions:", df.region.nunique())
print("Number of Verticals:", df.industry_vertical.nunique())
print("Distinct number of assigned reps:", df.assigned_rep_id.nunique())
print("Number of employee bands:", df.employee_band.nunique())
print("Number of loyalty tiers:", df.loyalty_tier.nunique())
df.describe().T[["min", "50%", "mean", "max"]].round(2)

(12000, 20)
duplicate ids: 0
nulls: 0
Number of Regions: 5
Number of Verticals: 6
Distinct number of assigned reps: 38
Number of employee bands: 4
Number of loyalty tiers: 4


,min,50%,mean,max
account_age_months,1.0,43.50,49.76,240.00
orders_12m,0.0,4.00,7.69,257.00
orders_last_90d,0.0,1.00,1.99,49.00
revenue_12m,0.0,5142.36,11497.18,435454.54
avg_order_value,0.0,1126.85,1458.13,28950.01
days_since_last_order,0.0,55.00,117.29,730.00
lifetime_orders,1.0,14.00,33.19,1691.00
consumables_revenue_share,0.0,0.53,0.53,1.00
email_open_rate_90d,0.0,0.40,0.41,0.99
portal_logins_90d,0.0,3.00,4.09,49.00


***
### **Notes**:
- 12,000 observations
- 0 duplicates
- 0 nulls
- 2 targets
- 12 historical features
- 5 regions
- 6 industry verticals
- 38 distinct assigned reps
- 4 company sizes
- 4 loyalty tiers
***

In [2]:
df.head()

,account_id,industry_vertical,region,assigned_rep_id,employee_band,loyalty_tier,account_age_months,orders_12m,orders_last_90d,revenue_12m,avg_order_value,days_since_last_order,lifetime_orders,consumables_revenue_share,email_open_rate_90d,portal_logins_90d,quote_requests_90d,rep_contacts_90d,ordered_next_30d,next_30d_revenue
0,ANT-10001,General Manufacturing,South Central,REP-31,251+,Platinum,99,10,0,38741.23,3874.12,264,308,0.548,0.579,16,2,8,0,0.00
1,ANT-10002,Machining & Auto Parts,Southeast,REP-18,1-10,Bronze,27,0,0,0.00,0.00,665,1,0.243,0.379,2,1,1,0,0.00
2,ANT-10003,Machining & Auto Parts,Northeast,REP-15,251+,Platinum,36,6,1,24980.00,4163.33,76,23,0.299,0.563,11,1,6,1,5922.33
3,ANT-10004,Plastics & Packaging,Southeast,REP-25,1-10,Bronze,4,3,0,1881.08,627.03,103,3,0.853,0.228,2,0,2,0,0.00
4,ANT-10005,Metal Fabrication,South Central,REP-27,51-250,Silver,83,7,0,5251.36,750.19,201,41,0.803,0.449,4,0,4,1,2022.44


## Business Alignment

In [3]:
# 2a. Internal consistency: each should print True
checks = {
    "orders_90d <= orders_12m": (df.orders_last_90d <= df.orders_12m).all(),
    "orders_12m <= lifetime_orders": (df.orders_12m <= df.lifetime_orders).all(),
    "revenue == 0 iff orders == 0": ((df.revenue_12m == 0) == (df.orders_12m == 0)).all(),
    "AOV ~= revenue / orders": np.allclose(
        df.avg_order_value,
        (df.revenue_12m / df.orders_12m.replace(0, np.nan)).fillna(0),
        atol=0.02,
    ),
    "next_30d_revenue > 0 iff ordered": ((df.next_30d_revenue > 0) == (df.ordered_next_30d == 1)).all(),
    "no-order accts have recency > 365": (df.loc[df.orders_12m == 0, "days_since_last_order"] > 365).all(),
}
pd.Series(checks)

orders_90d <= orders_12m             True
orders_12m <= lifetime_orders        True
revenue == 0 iff orders == 0         True
AOV ~= revenue / orders              True
next_30d_revenue > 0 iff ordered     True
no-order accts have recency > 365    True
dtype: bool

This is less of business alignment and more of EDA

In [4]:
# 2b. Scale vs. the email: 12K accounts, 38 reps, 25-30 touches/rep/week
reps = df.groupby("assigned_rep_id").size()
print("accounts:", len(df), "| reps:", len(reps), "| regions:", df.region.nunique())
print("accounts per rep (min/median/max):", reps.min(), reps.median(), reps.max())

cap_lo, cap_hi = 38 * 25, 38 * 30
print(f"weekly capacity: {cap_lo}-{cap_hi} touches = {cap_lo/len(df):.1%}-{cap_hi/len(df):.1%} of base")
print(f"base 30d purchase rate: {df.ordered_next_30d.mean():.1%}")

# Email says ~1/3 of accounts have < 3 orders
print("orders_12m < 3:", f"{(df.orders_12m < 3).mean():.1%}")
print("zero orders in 12m:", f"{(df.orders_12m == 0).mean():.1%}")

accounts: 12000 | reps: 38 | regions: 5
accounts per rep (min/median/max): 224 319.0 378
weekly capacity: 950-1140 touches = 7.9%-9.5% of base
base 30d purchase rate: 36.7%
orders_12m < 3: 35.4%
zero orders in 12m: 8.0%


8% of our accounts are lapsed.

In [5]:
# 2c. Do tiers, verticals, regions, and size behave the way sales would expect?
tier_order = ["Bronze", "Silver", "Gold", "Platinum"]
tiers = df.groupby("loyalty_tier").agg(
    n=("account_id", "size"),
    purchase_rate=("ordered_next_30d", "mean"),
    median_rev=("revenue_12m", "median"),
    median_orders=("orders_12m", "median"),
).loc[tier_order]
tiers["share"] = tiers.n / len(df)
display(tiers.round(3))

for col in ["industry_vertical", "region", "employee_band"]:
    display(
        df.groupby(col)
        .agg(n=("account_id", "size"), purchase_rate=("ordered_next_30d", "mean"), median_orders=("orders_12m", "median"))
        .round(3)
    )

,n,purchase_rate,median_rev,median_orders,share
loyalty_tier,,,,,
Bronze,5400,0.204,1679.375,2.0,0.45
Silver,3600,0.387,7076.730,5.0,0.30
Gold,2160,0.579,17348.265,10.0,0.18
Platinum,840,0.775,46193.035,21.0,0.07


,n,purchase_rate,median_orders
industry_vertical,,,
Construction & Contracting,1636,0.257,2.0
Food & Beverage Processing,1684,0.526,8.0
General Manufacturing,1908,0.327,3.0
Machining & Auto Parts,2392,0.396,5.0
Metal Fabrication,2672,0.317,4.0
Plastics & Packaging,1708,0.395,5.0


,n,purchase_rate,median_orders
region,,,
Great Lakes,3541,0.378,4.0
Northeast,2251,0.326,3.0
South Central,1886,0.423,5.0
Southeast,2528,0.389,5.0
West,1794,0.304,3.0


,n,purchase_rate,median_orders
employee_band,,,
1-10,4144,0.246,2.0
11-50,4197,0.350,4.0
251+,968,0.632,11.0
51-250,2691,0.483,6.0


I like this one, splitting out orders & purchase rate by categories to see if it makes sense.

1. Loyalty tier makes sense, more valuable but fewer accounts in the top tiers

In [6]:
# 2d. Recency: does "days since last order" behave the way the old rule assumes?
bins = [-1, 30, 60, 90, 180, 365, 730]
display(
    df.groupby(pd.cut(df.days_since_last_order, bins), observed=True)
    .agg(n=("account_id", "size"), purchase_rate=("ordered_next_30d", "mean"))
    .round(3)
)

# 2e. Consumables vs. capital: Terrence says these behave differently
display(
    df.groupby(pd.qcut(df.consumables_revenue_share, 4), observed=True)
    .agg(median_orders=("orders_12m", "median"),
         median_aov=("avg_order_value", "median"),
         purchase_rate=("ordered_next_30d", "mean"))
    .round(3)
)

,n,purchase_rate
days_since_last_order,,
"(-1, 30]",4333,0.569
"(30, 60]",2021,0.401
"(60, 90]",1193,0.325
"(90, 180]",1922,0.234
"(180, 365]",1573,0.133
"(365, 730]",958,0.077


,median_orders,median_aov,purchase_rate
consumables_revenue_share,,,
"(-0.001, 0.344]",2.0,1872.570,0.209
"(0.344, 0.533]",3.0,1367.740,0.317
"(0.533, 0.718]",5.0,1042.100,0.399
"(0.718, 1.0]",8.0,737.725,0.542


1. Recency is related to purchase rate, and you can see a bunch of accounts 90 to 365 days old that are due for a reach-out
2. Larger consumables revenue share are related to more orders, less order value, and higher purchase rate

In [7]:
# 2f. Engagement signals vs. outcome (quartiles, low to high)
for col in ["quote_requests_90d", "portal_logins_90d", "rep_contacts_90d", "email_open_rate_90d"]:
    q = pd.qcut(df[col].rank(method="first"), 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
    print(col, df.groupby(q, observed=True).ordered_next_30d.mean().round(3).to_dict())

quote_requests_90d {'Q1 low': 0.343, 'Q2': 0.344, 'Q3': 0.367, 'Q4 high': 0.413}
portal_logins_90d {'Q1 low': 0.212, 'Q2': 0.285, 'Q3': 0.381, 'Q4 high': 0.588}
rep_contacts_90d {'Q1 low': 0.24, 'Q2': 0.289, 'Q3': 0.366, 'Q4 high': 0.571}
email_open_rate_90d {'Q1 low': 0.338, 'Q2': 0.361, 'Q3': 0.378, 'Q4 high': 0.389}


1. All 4 of these features are positively related to 30 day orders, a few of these are levers we can pull

## EDA

In [8]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

y = df.ordered_next_30d

# 3a. Target balance
print(y.value_counts(normalize=True).round(3).to_dict())

{0: 0.633, 1: 0.367}


- 63.3% no orders in next 30 days
- 36.7% ordered in next 30 days

Binary Classification Model - skipping resampling via downsampling, SMOTE, or class weights because we care about calibration

In [9]:
# 3b. Skew and outliers: does log1p fix the long tails?
num_cols = [
    "account_age_months", "orders_12m", "orders_last_90d", "revenue_12m", "avg_order_value",
    "days_since_last_order", "lifetime_orders", "consumables_revenue_share", "email_open_rate_90d",
    "portal_logins_90d", "quote_requests_90d", "rep_contacts_90d",
]
pd.DataFrame({
    "skew_raw": df[num_cols].skew(),
    "skew_log1p": np.log1p(df[num_cols]).skew(),
    "p99": df[num_cols].quantile(0.99),
    "max": df[num_cols].max(),
}).round(2)

,skew_raw,skew_log1p,p99,max
account_age_months,1.12,-0.99,165.00,240.00
orders_12m,5.43,0.30,56.00,257.00
orders_last_90d,4.48,0.72,15.00,49.00
revenue_12m,6.03,-2.04,99174.05,435454.54
avg_order_value,3.06,-2.52,6296.81,28950.01
days_since_last_order,2.05,-0.42,686.00,730.00
lifetime_orders,6.74,0.25,287.01,1691.00
consumables_revenue_share,-0.06,-0.35,1.00,1.00
email_open_rate_90d,0.21,-0.07,0.82,0.99
portal_logins_90d,2.32,-0.07,19.00,49.00


Lots of skewed features - we can log them to fix it up.

In [10]:
# 3c. Collinearity (Spearman handles the skew)
corr = df[num_cols].corr(method="spearman").abs()
pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack().sort_values(ascending=False)
pairs[pairs > 0.7].round(2)

orders_last_90d  days_since_last_order    0.87
orders_12m       revenue_12m              0.86
                 lifetime_orders          0.77
                 orders_last_90d          0.75
dtype: float64

Heavy correlation between features here. Fine for trees, ok for regularization as well. For Bayesian, use tight priors & watch sampler.

In [11]:
# 3d. Hierarchy: is it clean, and how thin are the cells partial pooling would help?
print("reps in >1 region:", (df.groupby("assigned_rep_id").region.nunique() > 1).sum())
cell = df.groupby(["assigned_rep_id", "industry_vertical"]).size()
print("rep x vertical cell sizes:", cell.describe().round(0).to_dict())
print("thin accounts per rep:", df[df.orders_12m < 3].groupby("assigned_rep_id").size().agg(["min", "median", "max"]).to_dict())

reps in >1 region: 0
rep x vertical cell sizes: {'count': 228.0, 'mean': 53.0, 'std': 14.0, 'min': 24.0, '25%': 43.0, '50%': 50.0, '75%': 60.0, 'max': 94.0}
thin accounts per rep: {'min': 63.0, 'median': 116.5, 'max': 150.0}


Each rep is in 1 region, ignoring the rest of this because it's just EDA overkill

In [12]:
# 3e. Row-wise features (no leakage risk, so safe before the split)
exposure_days = np.minimum(df.account_age_months * 30.4, 365)       # days the account has existed, capped at 1 year
expected_gap = exposure_days / df.orders_12m.clip(lower=1)           # typical days between this account's orders
df["overdue_ratio"] = (df.days_since_last_order / expected_gap).clip(upper=20)  # >1 means later than usual
df["recent_share"] = df.orders_last_90d / df.orders_12m.clip(lower=1)           # share of 12m orders in last 90d
df["thin_flag"] = (df.orders_12m < 3).astype(int)
df["no_orders_12m"] = (df.orders_12m == 0).astype(int)

# 3f. Univariate AUC (0.5 = useless). Sign tells you direction
feats = num_cols + ["overdue_ratio", "recent_share"]
auc = pd.Series({c: roc_auc_score(y, df[c]) for c in feats})
pd.DataFrame({"auc": auc.round(3), "higher_means_more_likely": auc > 0.5}).sort_values("auc", key=lambda s: (s - 0.5).abs(), ascending=False)

,auc,higher_means_more_likely
orders_12m,0.781,True
orders_last_90d,0.750,True
revenue_12m,0.745,True
days_since_last_order,0.269,False
lifetime_orders,0.724,True
portal_logins_90d,0.679,True
rep_contacts_90d,0.655,True
consumables_revenue_share,0.654,True
recent_share,0.613,True
overdue_ratio,0.453,False


Feature engineering here
- exposure days = days the account has existed (max = 365)
- expected gap = days between accounts orders in past 12 months
- overdue ratio = day since last order / days between orders ~ are they due for an order? Could be an inverted-U (peak when slightly overdue, then churn).
- recent share = share of last years orders that were in the past 90 days
- thin flag = fewer than 3 orders, per Terrence on our email chain
- no orders past 12 m = lapsed essentially

Base Model
- Orders past 12m, ordrs past 90 days, revenue past 12m, days since last order, lifeitme orders are the best 5 features
- account age, email open rate, quote requests are the worst features


each day has the same order probability no matter how long it's been since the last order.

In [ ]:
bins = [0, 0.25, 0.5, 0.75, 1, 1.25, 1.5, 2, 3, 20]
df["overdue_bin"] = pd.cut(df.overdue_ratio, bins, include_lowest=True)
df["freq_band"] = pd.cut(df.orders_12m, [-1, 2, 5, 12, 1000], labels=["0-2", "3-5", "6-12", "13+"])

display(df.groupby("overdue_bin", observed=True).agg(n=("account_id", "size"), rate=("ordered_next_30d", "mean")).round(3))
display(df.pivot_table(index="overdue_bin", columns="freq_band", values="ordered_next_30d", aggfunc="mean", observed=True).round(3))

,n,rate
overdue_bin,,
"(-0.001, 0.25]",2165,0.441
"(0.25, 0.5]",1798,0.408
"(0.5, 0.75]",1600,0.393
"(0.75, 1.0]",1997,0.287
"(1.0, 1.25]",1128,0.333
"(1.25, 1.5]",896,0.324
"(1.5, 2.0]",1446,0.301
"(2.0, 3.0]",661,0.437
"(3.0, 20.0]",309,0.388


freq_band,0-2,3-5,6-12,13+
overdue_bin,,,,
"(-0.001, 0.25]",0.188,0.299,0.522,0.804
"(0.25, 0.5]",0.160,0.316,0.499,0.814
"(0.5, 0.75]",0.159,0.337,0.505,0.803
"(0.75, 1.0]",0.146,0.273,0.457,0.754
"(1.0, 1.25]",0.130,0.258,0.506,0.831
"(1.25, 1.5]",0.106,0.304,0.455,0.843
"(1.5, 2.0]",0.112,0.253,0.486,0.813
"(2.0, 3.0]",NaN,0.225,0.424,0.797
"(3.0, 20.0]",NaN,0.150,0.322,0.540


Overdue vs purchase rate and frequency

In [ ]:
# 3g. Split once (stratified on outcome x thin flag). Test set stays untouched until section 5
df["strata"] = df.ordered_next_30d.astype(str) + "_" + df.thin_flag.astype(str)
train, test = train_test_split(df, test_size=0.2, stratify=df.strata, random_state=42)
train = train.drop(columns="strata").reset_index(drop=True)
test = test.drop(columns="strata").reset_index(drop=True)

# # AOV for accounts with no orders: fill with the vertical x size median, fit on TRAIN only
# peer_keys = ["industry_vertical", "employee_band"]
# peer = (train[train.orders_12m > 0].groupby(peer_keys, as_index=False).avg_order_value.median()
#         .rename(columns={"avg_order_value": "aov_peer_median"}))
# train = train.merge(peer, on=peer_keys, how="left")
# test = test.merge(peer, on=peer_keys, how="left")
# for d in (train, test):
#     d["aov_filled"] = np.where(d.orders_12m > 0, d.avg_order_value, d.aov_peer_median)

print(train.shape, test.shape, train.ordered_next_30d.mean().round(3), test.ordered_next_30d.mean().round(3))

(9600, 26) (2400, 26) 0.367 0.367


1. Stratified sampling 80/20 on target & one thin account data (both binary)
2. hm...not sure how I feel about this...I"m ditching AOV Filled, no good.

In [14]:
# 3h. Feature lists for section 4
cat_features = ["industry_vertical", "region", "assigned_rep_id", "employee_band", "loyalty_tier"]
num_features = [c for c in num_cols if c != "avg_order_value"] + [
    "aov_filled", "overdue_ratio", "recent_share", "thin_flag", "no_orders_12m",
]
log_features = [  # log1p + standardize these for LR and Bayes. Trees can skip it
    "orders_12m", "orders_last_90d", "revenue_12m", "aov_filled", "days_since_last_order",
    "lifetime_orders", "portal_logins_90d", "quote_requests_90d", "rep_contacts_90d",
]
TARGET, VALUE_COL = "ordered_next_30d", "next_30d_revenue"  # value col is for the EV backtest only, never a feature

- preparing features for modeling & segmentation

## Segmentation

## Machine Learning

## Model Validation

## Insights